# Tuning - Random Forest Regressor para limiar de reposicao

Este notebook desenvolve o sucessor do artefato historico `03_tree_ensembles_random_forest_regressor_threshold_model.pkl`, mas sem salvar `.pkl` localmente. O modelo, metricas, predicoes, parametros e graficos sao registrados diretamente no MLflow.


## Estrategia

Usamos `TimeSeriesSplit`, uma validacao cruzada temporal. Ela preserva a ordem passado -> futuro e evita vazamento temporal, que ocorreria com K-Fold aleatorio. O conjunto `test` fica isolado para avaliacao final do campeao.

O tuning usa `RandomizedSearchCV` porque o grid completo seria caro para florestas com varias combinacoes de profundidade, folhas e numero de arvores. Alem dos hiperparametros, cada busca e repetida com perfis de peso diferentes para testar custos de erro diferentes.


In [ ]:
from pathlib import Path
import sys

import pandas as pd

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "regressor_tuning_common.py").exists():
    sys.path.insert(0, str(NOTEBOOK_DIR))
else:
    sys.path.insert(0, str((Path("data") / "notebooks").resolve()))

from regressor_tuning_common import (
    RegressorTuningConfig,
    WEIGHT_PROFILE_DESCRIPTIONS,
    run_regressor_tuning,
)

from sklearn.ensemble import RandomForestRegressor


In [ ]:
RF_PARAM_DISTRIBUTIONS = {
    "model__n_estimators": [100, 200, 300, 500],
    "model__max_depth": [None, 6, 10, 16, 24],
    "model__min_samples_split": [2, 5, 10, 20],
    "model__min_samples_leaf": [1, 2, 4, 8],
    "model__max_features": ["sqrt", "log2", 0.5, 0.8, 1.0],
    "model__bootstrap": [True],
}

def rf_model_factory() -> RandomForestRegressor:
    return RandomForestRegressor(random_state=42, n_jobs=-1)

def rf_baseline_factory() -> RandomForestRegressor:
    return RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)

config = RegressorTuningConfig(
    notebook_id="05_random_forest_regressor_threshold_tuning",
    family="Arvores",
    model_name="Random Forest Regressor",
    model_factory=rf_model_factory,
    baseline_factory=rf_baseline_factory,
    param_distributions=RF_PARAM_DISTRIBUTIONS,
    n_iter=16,
    cv_splits=4,
    random_state=42,
)

search_space = pd.DataFrame(
    [{"parametro": key, "valores": values} for key, values in RF_PARAM_DISTRIBUTIONS.items()]
)
weight_profiles = pd.DataFrame(
    [{"perfil": key, "descricao": value} for key, value in WEIGHT_PROFILE_DESCRIPTIONS.items()]
)

display(search_space)
display(weight_profiles)


## Execucao do tuning

A celula abaixo executa baseline, buscas randomicas por perfil de peso, registro dos candidatos no MLflow, treino final do campeao e diagnosticos de ajuste. Se o tempo estiver alto, reduza `n_iter` mantendo os mesmos perfis de peso.


In [ ]:
results = run_regressor_tuning(config)


## Resultados quantitativos

As tabelas abaixo mostram baseline, melhores candidatos, variancia entre folds e metricas finais no teste.


In [ ]:
display(results["baseline_summary"])
display(results["candidate_results"].head(15))
display(results["best_fold_metrics"])
display(results["final_metrics"].T)
print("Melhor perfil de peso:", results["best_weight_profile"])
print("Melhores hiperparametros:", results["best_params"])
print("Diagnostico:", results["fit_diagnosis"])


## Visualizacoes de ajuste

A curva de aprendizado ajuda a identificar overfitting ou underfitting. A analise de residuos mostra vieses e dispersao dos erros no limiar previsto.


In [ ]:
results["figures"]["learning_curve"]


In [ ]:
results["figures"]["residual_analysis"]


## MLflow

No MLflow, procure pelos runs do experimento `saltim_two_stage_05_random_forest_regressor_threshold_tuning_threshold_regression`. Os runs de candidatos guardam combinacoes de parametros e pesos; o run `champion` guarda o modelo registrado, predicoes, metricas finais e graficos.
